# Week 8 Presentation Brief — Variation A
## 🐱 Feral Cats and Numbats: Phase Plane Analysis
**SCIE1500**

> Work through all parts during the Week 8 lab. Your **10-minute Week 9 presentation** should cover: the ODE model, equilibrium analysis, phase plane interpretation, and conservation implications
> **What to submit:** every group member must individually upload their own copy of the same completed presentation slides (PDF or PowerPoint) to the LMS after your presentation — this lets your instructor verify who participated.


---
## 📋 Scenario: Does Cat Control Create Recovery?

![Detailed phase portrait with directional field and trajectory](../images/W8A_cat_control.svg)

A conservation team is considering cat-control measures to protect numbats. Before recommending an intervention, it needs to understand what the model predicts after populations are pushed away from their usual balance.

You will use the same Lotka-Volterra model as the main lab:

$$\frac{dN}{dt} = 0.4N - 0.002NC, \qquad \frac{dC}{dt} = -0.3C + 0.001NC$$

This variation moves beyond calculating one equilibrium. You will compare several starting conditions and ask whether each one returns to a stable state, cycles indefinitely, or creates a risky conservation trough.

**Decision question:** Is a low initial cat population enough to guarantee long-term numbat recovery? Your graphs should provide the evidence for your answer.

---
## 🎯 Your Task

| Part | Topic | Time |
|------|-------|------|
| A | Find equilibria and assess local stability | ~20 min |
| B | Plot a phase portrait with multiple trajectories using `odeint` | ~25 min |
| C | Identify population thresholds and conservation risks | ~15 min |

**Presentation preparation:** Give every member ownership of one claim. For each claim, name the equation or graph feature that supports it; do not report a result without interpreting its ecological meaning.

In [ ]:
# Run first — loads libraries for this session
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
from scipy.integrate import odeint

r_N = 0.4; a = 0.002; d = 0.3; b = 0.001

def lotka_volterra(state, t):
    N, C = state
    dN = r_N*N - a*N*C
    dC = -d*C  + b*N*C
    return [dN, dC]

✏️ **Part A — Find and interpret the coexistence equilibrium:** set $dN/dt = 0$ and $dC/dt = 0$, and solve for $N^*$ and $C^*$.

Do not treat these as two isolated numbers. At coexistence, each population is exactly at the threshold where its own instantaneous growth changes sign: $C^*$ is the cat level that makes numbat growth zero, while $N^*$ is the numbat level that makes cat growth zero.

**Presentation prompt:** Say what $(N^*, C^*) = (300, 200)$ means in a full sentence, then explain why it is a useful reference point even when a simulated trajectory does not remain there. Run the cell below to check your algebra.

In [ ]:
# Coexistence equilibrium: dN/dt=0 → C*=r_N/a=200; dC/dt=0 → N*=d/b=300
N_star = d / b   # 300
C_star = r_N / a  # 200

print(f"Equilibrium: N*={N_star:.0f}, C*={C_star:.0f}")

Multiple trajectories from different initial conditions.

In [ ]:
# Multiple trajectories from different initial conditions
t_span = np.linspace(0, 40, 2000)
initial_conditions = [
    (400, 80),   # high numbats, low cats
    (200, 250),  # low numbats, high cats
    (350, 150),  # moderate both
    (500, 200),  # very high numbats
]

colors = ["steelblue", "firebrick", "seagreen", "purple"]
labels = ["N₀=400, C₀=80", "N₀=200, C₀=250", "N₀=350, C₀=150", "N₀=500, C₀=200"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: phase plane
N_grid = np.linspace(0, 700, 18)
C_grid = np.linspace(0, 350, 18)
NN, CC = np.meshgrid(N_grid, C_grid)
dN_dt = r_N*NN - a*NN*CC
dC_dt = -d*CC  + b*NN*CC
mag = np.sqrt(dN_dt**2 + dC_dt**2) + 1e-6
axes[0].quiver(NN, CC, dN_dt/mag, dC_dt/mag, mag, cmap="Greys", alpha=0.5)

for ic, col, lbl in zip(initial_conditions, colors, labels):
    sol = odeint(lotka_volterra, ic, t_span)
    axes[0].plot(sol[:, 0], sol[:, 1], col, lw=2, label=lbl)
    axes[0].plot(ic[0], ic[1], "o", color=col, ms=8)

axes[0].axvline(N_star, color="k", ls="--", alpha=0.5, label=f"N*={N_star:.0f}")
axes[0].axhline(C_star, color="k", ls=":",  alpha=0.5, label=f"C*={C_star:.0f}")
axes[0].plot(N_star, C_star, "k*", ms=16, zorder=6, label="Equilibrium")
axes[0].set_xlabel("Numbats N"); axes[0].set_ylabel("Cats C")
axes[0].set_title("Phase Portrait"); axes[0].legend(fontsize=7)
axes[0].set_xlim(0, 700); axes[0].set_ylim(0, 350)

# Right: time series for first IC
sol0 = odeint(lotka_volterra, initial_conditions[0], t_span)
axes[1].plot(t_span, sol0[:, 0], "steelblue", lw=2, label="Numbats")
axes[1].plot(t_span, sol0[:, 1], "firebrick", lw=2, label="Feral Cats")
axes[1].axhline(N_star, color="steelblue", ls="--", alpha=0.4)
axes[1].axhline(C_star, color="firebrick", ls="--", alpha=0.4)
axes[1].set_xlabel("Years"); axes[1].set_ylabel("Population")
axes[1].set_title("Time Series (N₀=400, C₀=80)"); axes[1].legend()

plt.tight_layout(); plt.show()

## Part C: From a Trajectory to a Conservation Risk

The trajectories differ only in their starting populations. Compare their **amplitude**: how far each path moves above and below the coexistence point. A large swing may matter more for conservation than an attractive-looking starting value, because it can produce a later low numbat population.

**Prompt before running the code:** Predict which starting condition is most likely to produce the lowest numbat population. Then report the minimum numbat population from the simulation and explain why this matters for a management decision.

Be precise with the threshold: when $N < 300$, the cat population has $dC/dt < 0$ **at that moment**. That sign tells you the local direction of change; it does not by itself prove that cats will become extinct in the coupled model.

In [ ]:
# Conservation threshold: if N drops below what level do cats go extinct?
# Cats go extinct if dC/dt < 0 always → N < d/b = 300 at all times
# Find minimum N across all trajectories
for ic, lbl in zip(initial_conditions, labels):
    sol = odeint(lotka_volterra, ic, t_span)
    N_min = sol[:, 0].min()
    C_max = sol[:, 1].max()
    print(f"{lbl}: N_min={N_min:.0f}, C_max={C_max:.0f}")

---
## ✅ Presentation Checklist (Week 9, 10 minutes)

1. **Model** (~2 min): State the ODEs; explain each parameter ecologically.
2. **Equilibrium** (~2 min): Derive $N^* = 300$, $C^* = 200$; interpret the nullclines.
3. **Phase portrait** (~4 min): Show the plot; trace one trajectory; explain the closed-orbit pattern.
4. **Conservation** (~2 min): What starting conditions lead to cat dominance vs. numbat recovery?

---
## 📊 Presentation Marking Rubric (20 marks → scaled to 4% of your unit grade)

Your group presentation is graded out of 20 marks (scaled to 4% of your unit grade — each group presents twice, for 8% total). This rubric determines your group mark, which will be the mark for each contributing member unless we are advised otherwise.

| Criterion | Excellent | Good | Developing | Poor |
|---|---|---|---|---|
| **Problem Formulation** (5 marks) | Clear explanation of the real-world problem; audience understands what question is being answered and why it matters (5) | Problem explained but lacks full context or motivation (3–4) | Problem stated but unclear why it's important (1–2) | No clear problem statement (0) |
| **Mathematical Approach** (5 marks) | Correct model/method selected; clear justification for the choice; key equations presented clearly (5) | Correct approach with minor errors; justification present but weak (3–4) | Approach has errors or is poorly justified (1–2) | Wrong method or no mathematical content shown (0) |
| **Results & Interpretation** (6 marks) | Results are correct and clearly presented; findings are connected to a real-world decision, including limitations/trade-offs (6) | Results mostly correct; some interpretation but lacks depth (4–5) | Results unclear, minor errors, or interpretation is minimal — just states numbers (2–3) | Major errors, no results shown, or no interpretation given (0–1) |
| **Communication Quality** (4 marks) | Effective graphs/visuals support the story; all members participate and speak without reading from notes; well-rehearsed and within the 10-minute limit (4) | Visuals adequate; most members participate; slightly over/under time (3) | Visualization ineffective or missing; uneven participation; timing issues (1–2) | No visuals; one person dominates; major timing problems (0) |

**Total: ____ / 20 marks**